In [1]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights
from PIL import Image
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score

# --- 1. DATASET ---
class WikiArtFilteredDataset(Dataset):
    def __init__(self, csv_file, split, transform=None):
        self.df = pd.read_csv(csv_file)
        self.df = self.df[self.df['split'] == split].reset_index(drop=True)
        self.transform = transform
        self.classes = sorted(self.df['label'].unique())
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        label = self.class_to_idx[self.df.loc[idx, 'label']]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Běží na: {device}")
    
    # Dynamická augmentace vč. Random Erasing pro Milestone 2
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.2)
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    csv_path = "dataset_filtered_split.csv"
    train_dataset = WikiArtFilteredDataset(csv_path, 'train', train_transform)
    val_dataset = WikiArtFilteredDataset(csv_path, 'val', val_transform)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

    # --- 2. VÝPOČET INVERZNÍCH VAH ---
    print("Počítám inverzní váhy tříd...")
    train_labels_idx = [train_dataset.class_to_idx[lbl] for lbl in train_dataset.df['label']]
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_labels_idx),
        y=train_labels_idx
    )
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

    # --- 3. PŘÍPRAVA MODELU A OPTIMALIZACE ---
    model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    
    # Zmrazení základních vrstev
    for param in model.parameters():
        param.requires_grad = False
        
    model.fc = nn.Linear(model.fc.in_features, len(train_dataset.classes))
    model = model.to(device)

    # CrossEntropyLoss nyní tvrdě penalizuje chyby na vzácných třídách
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    
    # AdamW obsahuje Weight Decay
    optimizer = optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)
    
    epochs = 10
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-3, steps_per_epoch=len(train_loader), epochs=epochs)
    writer = SummaryWriter('runs/wikiart_filtered_frozen')

    best_val_f1 = 0.0
    print("Spouštím trénování (Fáze 1 - Zmrazený model)...")

    # --- 4. TRÉNOVACÍ SMYČKA ---
    for epoch in range(epochs):
        start_time = time.time()
        
        # -- TRAIN --
        model.train()
        running_train_loss = 0.0
        train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        
        for inputs, labels in train_loop:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()
            
            running_train_loss += loss.item() * inputs.size(0)
            train_loop.set_postfix(loss=running_train_loss/len(train_dataset))
            
        epoch_train_loss = running_train_loss / len(train_dataset)
        
        # -- VAL (Macro F1) --
        model.eval()
        running_val_loss = 0.0
        all_preds = []
        all_targets = []
        
        val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]  ")
        with torch.no_grad():
            for inputs, labels in val_loop:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                running_val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(labels.cpu().numpy())
                
        epoch_val_loss = running_val_loss / len(val_dataset)
        epoch_val_f1 = f1_score(all_targets, all_preds, average='macro')
        
        writer.add_scalar('Loss/Train', epoch_train_loss, epoch)
        writer.add_scalar('Loss/Validation', epoch_val_loss, epoch)
        writer.add_scalar('F1/Validation_Macro', epoch_val_f1, epoch)
        
        if epoch_val_f1 > best_val_f1:
            best_val_f1 = epoch_val_f1
            torch.save(model.state_dict(), 'best_resnet18_filtered_frozen.pth')
            saved_msg = " -> Uloženo (Nový rekord!)"
        else:
            saved_msg = ""
            
        print(f"Shrnutí {epoch+1} [{time.time() - start_time:.0f}s]: Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Macro F1: {epoch_val_f1:.4f}{saved_msg}\n")

if __name__ == "__main__":
    main()

Běží na: cuda
Počítám inverzní váhy tříd...
Spouštím trénování (Fáze 1 - Zmrazený model)...


Epoch 1/10 [Val]  : 100%|██████████| 222/222 [01:15<00:00,  2.93it/s]


Shrnutí 1 [594s]: Train Loss: 2.1624 | Val Loss: 1.8035 | Val Macro F1: 0.3713 -> Uloženo (Nový rekord!)



Epoch 2/10 [Val]  : 100%|██████████| 222/222 [01:15<00:00,  2.95it/s]


Shrnutí 2 [580s]: Train Loss: 1.8794 | Val Loss: 1.8284 | Val Macro F1: 0.3605



Epoch 3/10 [Val]  : 100%|██████████| 222/222 [01:14<00:00,  2.99it/s]


... (11 outputs left out)

Epoch 9/10 [Val]  : 100%|██████████| 222/222 [01:15<00:00,  2.96it/s]


Shrnutí 9 [589s]: Train Loss: 1.7011 | Val Loss: 1.6131 | Val Macro F1: 0.4164



Epoch 10/10 [Val]  : 100%|██████████| 222/222 [01:11<00:00,  3.10it/s]

Shrnutí 10 [556s]: Train Loss: 1.6751 | Val Loss: 1.6043 | Val Macro F1: 0.4177



In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter

# 1. Znovuvytvoření globálního prostředí
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2)
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

csv_path = "dataset_filtered_split.csv"
# Třída WikiArtFilteredDataset zůstala v paměti z minulé buňky
train_dataset = WikiArtFilteredDataset(csv_path, 'train', train_transform)
val_dataset = WikiArtFilteredDataset(csv_path, 'val', val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

train_labels_idx = [train_dataset.class_to_idx[lbl] for lbl in train_dataset.df['label']]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels_idx),
    y=train_labels_idx
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# 2. Znovuvytvoření modelu a NAČTENÍ VAH z Fáze 1
model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, len(train_dataset.classes))
model = model.to(device)

model.load_state_dict(torch.load('best_resnet18_filtered_frozen.pth', weights_only=True))
print("Nejlepší váhy ze zmrazené fáze úspěšně načteny.")

# 3. Postupné rozmrazování (Gradual Unfreezing)
for param in model.parameters():
    param.requires_grad = False  # Zmrazit vše
for param in model.layer3.parameters():
    param.requires_grad = True   # Odemknout layer3
for param in model.layer4.parameters():
    param.requires_grad = True   # Odemknout layer4
for param in model.fc.parameters():
    param.requires_grad = True   # Odemknout klasifikační hlavu

# 4. Diskriminativní learning raty
optimizer = optim.AdamW([
    {'params': model.layer3.parameters(), 'lr': 1e-5},
    {'params': model.layer4.parameters(), 'lr': 5e-5}, 
    {'params': model.fc.parameters(), 'lr': 1e-4}
], weight_decay=1e-4)

epochs_unfrozen = 10
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[1e-4, 5e-4, 1e-3],
    steps_per_epoch=len(train_loader),
    epochs=epochs_unfrozen
)

writer = SummaryWriter('runs/wikiart_filtered_unfrozen')
best_val_f1_unfrozen = 0.0

print("Spouštím Fázi 2: Fine-tuning s rozmrazenými bloky...")

# 5. Trénovací smyčka
for epoch in range(epochs_unfrozen):
    start_time = time.time()
    
    # -- TRAIN --
    model.train()
    running_train_loss = 0.0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs_unfrozen} [Train]")
    
    for inputs, labels in train_loop:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        
        running_train_loss += loss.item() * inputs.size(0)
        train_loop.set_postfix(loss=running_train_loss/len(train_dataset))
        
    epoch_train_loss = running_train_loss / len(train_dataset)
    
    # -- VAL --
    model.eval()
    running_val_loss = 0.0
    all_preds = []
    all_targets = []
    
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs_unfrozen} [Val]  ")
    with torch.no_grad():
        for inputs, labels in val_loop:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    epoch_val_loss = running_val_loss / len(val_dataset)
    epoch_val_f1 = f1_score(all_targets, all_preds, average='macro')
    
    writer.add_scalar('Loss/Train_Unfrozen', epoch_train_loss, epoch)
    writer.add_scalar('Loss/Validation_Unfrozen', epoch_val_loss, epoch)
    writer.add_scalar('F1/Validation_Macro_Unfrozen', epoch_val_f1, epoch)
    
    if epoch_val_f1 > best_val_f1_unfrozen:
        best_val_f1_unfrozen = epoch_val_f1
        torch.save(model.state_dict(), 'best_resnet18_filtered_final.pth')
        saved_msg = " -> Uloženo (Nový rekord!)"
    else:
        saved_msg = ""
        
    print(f"Shrnutí Fáze 2 - {epoch+1} [{time.time() - start_time:.0f}s]: Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Macro F1: {epoch_val_f1:.4f}{saved_msg}\n")

writer.close()

NameError: name 'model' is not defined